# Task 2 — Modelling at Scale

| | |
|---|---|
| **Module** | 7006SCN Machine Learning and Big Data |
| **Student** | Manish Saini — SID 17142113 — sainim10@coventry.ac.uk |
| **Pool reference** | 112 |
| **Dataset** | NOAA GHCN-Daily, US stations (`CatLikeIceCream/temperature`, `climate.csv`), prepared in Task 1 |
| **Problem** | Regression: predict `label_TMAX_next`, the next-day maximum temperature (°C) at the same station |

 
*AI use opous 3:* cells marked `## AI:` i used  ai to undersand the concept  and logics of  the code  even  asked for  extra methods  which  help me to  gain good marks  also  used  to  solve  errors   i mension  everthing  code refrence and  ai  used .


# Environment, paths and run settings
These are the same Windows settings as Task 1. Spark needs the Hadoop helper files (`winutils.exe`, `hadoop.dll`) from `E:\hadoop`, and its temporary files go to `E:` because `C:` is almost full.

All paths are **relative to the repository root**. The notebook finds the root by looking upwards from its own folder for `results/task1.json`.

Every setting that affects the results is fixed here, before any model is trained.

In [1]:
# ==============================================================================
# 0.1  Environment, paths and run settings
# REF: Task 1 notebook, cells 0.1 and 2 - the same Windows/Hadoop and temp-folder settings are reused
## AI: project-root search and output-folder layout written with AI assistance
# ==============================================================================
import os, sys, json, time, math, platform, tempfile, statistics, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# ---- Windows only: Hadoop helper files and a temp folder that is not on the full C: drive ----
if os.name == "nt":
    os.environ["HADOOP_HOME"] = os.environ.get("HADOOP_HOME", r"E:\hadoop")
    os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]
    LOCAL_DIR = os.environ.get("CW_LOCAL_DIR", "E:/spark-tmp")
    os.environ["TEMP"] = os.environ["TMP"] = LOCAL_DIR
else:
    LOCAL_DIR = os.environ.get("CW_LOCAL_DIR", os.path.join(tempfile.gettempdir(), "spark-tmp"))
os.makedirs(LOCAL_DIR, exist_ok=True)


def find_project_root():
    """The repository root: the nearest folder (this one or above) whose results/task1.json has Task 1's
    'handoff' block AND whose handoff data exists. notebooks/results/task1.json is an older copy without
    the handoff, so it is skipped even if this notebook is opened from notebooks/."""
    tried = []
    for candidate in (Path("."), Path(".."), Path("../..")):
        f = candidate / "results" / "task1.json"
        if f.exists():
            tried.append(f.resolve().as_posix())
            h = json.loads(f.read_text()).get("handoff")
            if h and (candidate / h["train_path"]).exists():
                return candidate
    raise FileNotFoundError(f"No results/task1.json with a 'handoff' block was found (checked: {tried}). "
                            "Run the handoff cell of notebooks/task1.ipynb, then open this notebook from "
                            "the task2/ folder of the repository.")


ROOT = find_project_root()            # relative path to the repository root ("..", when run from task2/)

# ---- run settings: all fixed before any model is trained ----
QUICK_TEST      = False   # True = short smoke test on small samples; outputs go to task2/quick_test/ (not results)
SEED            = 42
TUNE_PCT        = 2 if QUICK_TEST else 10     # % of training rows used for cross-validation (fixed hash sample)
FINAL_TRAIN_PCT = 5 if QUICK_TEST else 100    # % of training rows used to refit the winning configuration
NUM_FOLDS       = 3                            # three blocks of consecutive years (section 1.7)
CV_PARALLELISM  = 2                            # parameter combinations evaluated at the same time (16 GB RAM)
PLOT_PER_MILLE  = 12                           # about 1.2% of test rows (~5,000 days) are drawn in the plots
INFER_REPEATS   = 3                            # inference is timed 3 times; the median is reported
REUSE_SAVED     = True                         # after a crash, models that already finished are loaded, not refitted

# ---- output folders 
OUT_DIR   = ROOT / "task2" / ("quick_test" if QUICK_TEST else "")
RES_DIR   = OUT_DIR / "results"
FIG_DIR   = OUT_DIR / "figures"
CKPT_DIR  = RES_DIR / "checkpoints"
MODEL_DIR = ROOT / "models" / ("task2_quick_test" if QUICK_TEST else "task2")   # models/ is git-ignored
for d in (RES_DIR, FIG_DIR, CKPT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)


def spark_path(p):
    """Spark gets an absolute path; everything printed or saved stays relative to the repository root."""
    return Path(p).resolve().as_posix()


def rel(p):
    return Path(os.path.relpath(Path(p).resolve(), ROOT.resolve())).as_posix()


def show(df, title=None):
    """Display a pandas table with a title (HTML in Jupyter, plain text elsewhere)."""
    if title:
        print(title)
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string(index=False))


print(f"repository root : {ROOT.resolve()}  (Task 1 contract: {rel(ROOT / 'results' / 'task1.json')})")
print(f"outputs         : {rel(RES_DIR)}/  {rel(FIG_DIR)}/  {rel(OUT_DIR / 'process_note.md')}")
print(f"fitted models   : {rel(MODEL_DIR)}/  (git-ignored)")
print(f"Spark temp dir  : {LOCAL_DIR}")
print(f"mode            : {'QUICK TEST - small samples, not for the report' if QUICK_TEST else 'full run'}")

repository root : C:\Users\Manish Saini\sainim10_17142113  (Task 1 contract: results/task1.json)
outputs         : task2/results/  task2/figures/  task2/process_note.md
fitted models   : models/task2/  (git-ignored)
Spark temp dir  : E:/spark-tmp
mode            : full run


### 2.0 SparkSession and executor configuration
This uses the same configuration as Task 1:

- `local[*]`, so all laptop cores are used;
- a 4 GB driver (6 GB crashed the JVM in Task 1);
- 32 shuffle partitions;
- adaptive query execution and ANSI mode switched on.

The configuration is recorded because Task 2 compares accuracy against compute cost.

In [2]:
# ==============================================================================
#  SparkSession and executor configuration
# REF: Task 1 notebook section 2 (SparkSession); Day05_LiveCoding_Colab > Block 0 (cell-2, cell-3)
#      Changed: same settings as Task 1, plus a checkpoint folder that GBTRegressor uses.
# ==============================================================================
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark import SparkContext
from pyspark.sql import SparkSession, functions as F

# clear any dead Spark/Java session left over from a crash (same block as Task 1)
try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass
SparkContext._active_spark_context = None; SparkContext._gateway = None; SparkContext._jvm = None
SparkSession._instantiatedSession = None; SparkSession._activeSession = None

DRIVER_MEMORY      = os.environ.get("CW_DRIVER_MEMORY", "4g")
SHUFFLE_PARTITIONS = int(os.environ.get("CW_SHUFFLE_PARTITIONS", "32"))

builder = (SparkSession.builder
           .appName("7006SCN-Task2-Modelling")
           .master("local[*]")                                         # EXPLAIN: one JVM using every CPU core
           .config("spark.driver.memory", DRIVER_MEMORY)
           .config("spark.local.dir", LOCAL_DIR)                       # EXPLAIN: shuffle and spill files
           .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
           .config("spark.sql.adaptive.enabled", "true")
           .config("spark.sql.ansi.enabled", "true")
           .config("spark.ui.showConsoleProgress", "false"))
if os.name == "nt":                                                    # EXPLAIN: lets the JVM load hadoop.dll
    hh = os.environ["HADOOP_HOME"].replace("\\", "/")
    builder = builder.config("spark.driver.extraJavaOptions",
                             f"-Djava.library.path={hh}/bin -Dhadoop.home.dir={hh}")
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
sc.setCheckpointDir(Path(LOCAL_DIR, "checkpoints").as_posix())         # EXPLAIN: GBT cuts its long lineage here

EXECUTOR_CONFIG = {
    "spark_version": spark.version,
    "master": sc.master,
    "cores (default parallelism)": sc.defaultParallelism,
    "driver_memory": spark.conf.get("spark.driver.memory"),
    "executor": "local mode - the driver JVM is also the executor",
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "cv_parallelism": CV_PARALLELISM,
    "os": platform.platform(),
    "python": platform.python_version(),
}
EXECUTOR_STR = (f"{sc.master}, {sc.defaultParallelism} cores, driver {spark.conf.get('spark.driver.memory')}, "
                f"{SHUFFLE_PARTITIONS} shuffle partitions, CV parallelism {CV_PARALLELISM}")
for k, v in EXECUTOR_CONFIG.items():
    print(f"{k:<28}: {v}")
print("Spark UI:", sc.uiWebUrl)

spark_version               : 4.2.0
master                      : local[*]
cores (default parallelism) : 12
driver_memory               : 4g
executor                    : local mode - the driver JVM is also the executor
shuffle_partitions          : 32
cv_parallelism              : 2
os                          : Windows-11-10.0.26300-SP0
python                      : 3.14.6
Spark UI: http://10.1.85.165:4040


## setup2.2 Data preparation at scale

Task 1 has already done the heavy work:

- it ingested 17.1 million rows;
- it reshaped them to one row per station per day;
- it built the lag, rolling and seasonal features;
- it split the data by year;
- it fitted the preprocessing pipeline on the training years.

### 2.2.1 Load the Task 1 DataFrames

In [3]:
# ==============================================================================
# 1.1  Load the DataFrames and the pipeline that Task 1 prepared
# REF: Day05_LiveCoding_Colab > Block 4 (cell-11): PipelineModel.load(path) and reuse without refitting
# ==============================================================================
from pyspark.ml import Pipeline, PipelineModel

task1 = json.loads((ROOT / "results" / "task1.json").read_text())
assert "handoff" in task1, "results/task1.json has no 'handoff' block - re-run the handoff cell of Task 1"
H = task1["handoff"]
TARGET = H["target_col"]                                   # label_TMAX_next
for k in ("train_path", "test_path", "preprocessing_model_path"):
    assert (ROOT / H[k]).exists(), f"Missing Task 1 output: {H[k]}"

train_raw = spark.read.parquet(spark_path(ROOT / H["train_path"]))
test_raw  = spark.read.parquet(spark_path(ROOT / H["test_path"]))
t1_prep   = PipelineModel.load(spark_path(ROOT / H["preprocessing_model_path"]))

print(f"train data    : {H['train_path']}  ({train_raw.rdd.getNumPartitions()} partitions)")
print(f"test data     : {H['test_path']}  ({test_raw.rdd.getNumPartitions()} partitions)")
print(f"preprocessing : {H['preprocessing_model_path']}  {[type(s).__name__ for s in t1_prep.stages]}")
print(f"target        : {TARGET} = {task1['problem']['target_meaning']}")
train_raw.printSchema()

train data    : lakehouse/handoff/train  (11 partitions)
test data     : lakehouse/handoff/test  (12 partitions)
preprocessing : models/task1_preprocessing  ['ImputerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel']
target        : label_TMAX_next = next-day maximum temperature at the same station (deg C)
root
 |-- ID: string (nullable = true)
 |-- date: date (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- state_code: string (nullable = true)
 |-- label_TMAX_next: double (nullable = true)
 |-- TMAX: double (nullable = true)
 |-- TMIN: double (nullable = true)
 |-- TOBS: double (nullable = true)
 |-- PRCP: double (nullable = true)
 |-- SNOW: double (nullable = true)
 |-- SNWD: double (nullable = true)
 |-- TMAX_lag1: double (nullable = true)
 |-- TMAX_lag2: double (nullable = true)
 |-- TMAX_roll7: double (nullable = true)
 |-- TMIN_roll7: double (nullable = true)
 |-- DTR: double (nullable = true)
 |-- doy_sin: double (nullable = 

### 2.0.1 Feature list and data types

In [4]:
# ==============================================================================
# 1.2  Feature list and data types
## AI: schema-check table written with AI assistance
# ==============================================================================
imp_m, idx_m, ohe_m, va_t1, scaler_m = t1_prep.stages      # Task 1's five fitted stages
NUM_FEATURES = list(imp_m.getInputCols())                  # numeric features (median-imputed in Task 1)
CAT_FEATURES = list(idx_m.getInputCols())                  # ['state_code'] (indexed, then one-hot encoded)
KEY_COLS     = ["ID", "date", "YEAR"]                      # identify a row and the split; never model inputs
FEATURES_COL = scaler_m.getOutputCol()                     # 'features' - the vector every model reads

expected = {"ID": "string", "date": "date", "YEAR": "int", **{c: "string" for c in CAT_FEATURES},
            **{c: "double" for c in NUM_FEATURES}, TARGET: "double"}
train_types, test_types = dict(train_raw.dtypes), dict(test_raw.dtypes)


def role(c):
    if c == TARGET:
        return "target"
    if c in NUM_FEATURES:
        return "numeric feature"
    return "categorical feature" if c in CAT_FEATURES else "row key"


types_df = pd.DataFrame([{"column": c, "role": role(c), "expected": t,
                          "train": train_types.get(c), "test": test_types.get(c),
                          "ok": train_types.get(c) == t and test_types.get(c) == t}
                         for c, t in expected.items()])
show(types_df, "Table 2.0a  Column types: expected vs actual")
assert types_df["ok"].all(), "A column is missing or has an unexpected type - see the table above"
unused = sorted(set(train_raw.columns) - set(expected))
print(f"{len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical feature columns; "
      f"columns not used: {unused if unused else 'none'}")
types_df.to_csv(RES_DIR / "task2_data_types.csv", index=False)

Table 2.0a  Column types: expected vs actual


,column,role,expected,train,test,ok
0,ID,row key,string,string,string,True
1,date,row key,date,date,date,True
2,YEAR,row key,int,int,int,True
3,state_code,categorical feature,string,string,string,True
4,TMAX,numeric feature,double,double,double,True
5,TMIN,numeric feature,double,double,double,True
6,TOBS,numeric feature,double,double,double,True
7,PRCP,numeric feature,double,double,double,True
8,SNOW,numeric feature,double,double,double,True
9,SNWD,numeric feature,double,double,double,True


14 numeric + 1 categorical feature columns; columns not used: none


### 2.0.2 Data validation
Before any model sees the data, these checks must pass:

- the row counts match the Task 1 contract;
- the training years end before the test years start;
- there are no duplicate station-days;
- the target has no missing values and stays inside Task 1's validity range of −60 to 60 °C;
- no target or future column appears among the features.


In [5]:
# ==============================================================================
#   Data validation: rows, split, duplicates, target and leakage
## AI: validation checks written with AI assistance
# ==============================================================================
checks = []


def check(name, passed, detail):
    checks.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")


n_train, n_test = train_raw.count(), test_raw.count()
check("Row counts match Task 1", n_train == H["train_rows"] and n_test == H["test_rows"],
      f"train {n_train:,} (Task 1: {H['train_rows']:,}), test {n_test:,} (Task 1: {H['test_rows']:,})")

yr_tr = train_raw.agg(F.min("YEAR"), F.max("YEAR")).first()
yr_te = test_raw.agg(F.min("YEAR"), F.max("YEAR")).first()
check("Training years end before test years start", yr_tr[1] < yr_te[0],
      f"train {yr_tr[0]}-{yr_tr[1]}, test {yr_te[0]}-{yr_te[1]}")

for split, df, n in (("train", train_raw, n_train), ("test", test_raw, n_test)):
    dups = n - df.select("ID", "date").distinct().count()
    check(f"No duplicate station-days ({split})", dups == 0, f"{dups:,} duplicate (ID, date) rows")

tgt = train_raw.unionByName(test_raw).agg(F.sum(F.col(TARGET).isNull().cast("int")).alias("nulls"),
                                          F.min(TARGET).alias("lo"), F.max(TARGET).alias("hi")).first()
check("Target has no missing values", tgt["nulls"] == 0, f"{tgt['nulls']:,} missing")
check("Target inside Task 1's valid range (-60 to 60 deg C)", -60 <= tgt["lo"] and tgt["hi"] <= 60,
      f"min {tgt['lo']:.1f}, max {tgt['hi']:.1f} deg C")

leaky = [c for c in NUM_FEATURES + CAT_FEATURES if c == TARGET or "next" in c.lower() or "lead" in c.lower()]
check("No target or future column among the features", not leaky,
      f"{len(NUM_FEATURES + CAT_FEATURES)} feature columns checked" if not leaky else f"leaky: {leaky}")

train_states = {r[0] for r in train_raw.select(CAT_FEATURES[0]).distinct().collect()}
test_states  = {r[0] for r in test_raw.select(CAT_FEATURES[0]).distinct().collect()}
check("Unseen categories in test are handled", idx_m.getHandleInvalid() == "keep",
      f"{len(train_states)} {CAT_FEATURES[0]} values in train, {len(test_states)} in test, "
      f"{len(test_states - train_states)} unseen (StringIndexer handleInvalid='{idx_m.getHandleInvalid()}')")

pd.DataFrame(checks).to_csv(RES_DIR / "task2_validation_checks.csv", index=False)
assert all(c["passed"] for c in checks), "A validation check failed - fix it before modelling"

[PASS] Row counts match Task 1: train 2,555,512 (Task 1: 2,555,512), test 433,788 (Task 1: 433,788)
[PASS] Training years end before test years start: train 1970-2015, test 2016-2023
[PASS] No duplicate station-days (train): 0 duplicate (ID, date) rows
[PASS] No duplicate station-days (test): 0 duplicate (ID, date) rows
[PASS] Target has no missing values: 0 missing
[PASS] Target inside Task 1's valid range (-60 to 60 deg C): min -53.9, max 50.0 deg C
[PASS] No target or future column among the features: 15 feature columns checked
[PASS] Unseen categories in test are handled: 40 state_code values in train, 38 in test, 0 unseen (StringIndexer handleInvalid='keep')


### 2.0.3 Missing values


In [6]:
# ==============================================================================
#   Missing values: measured in one pass per split, handled inside the pipeline
# REF: Week1_Day02_PySpark_Student_Workbook > Exercise 3, Sub-task 3.1 (completeness profile)
# ==============================================================================
def missing_pct(df, cols):
    n = df.count()
    exprs = []
    for c in cols:
        miss = F.col(c).isNull()
        if c in NUM_FEATURES or c == TARGET:
            miss = miss | F.isnan(c)                       # EXPLAIN: NaN counts as missing for doubles
        exprs.append((F.sum(miss.cast("int")) / n * 100).alias(c))
    return df.select(exprs).first().asDict()


cols = CAT_FEATURES + NUM_FEATURES + [TARGET]
miss_tr, miss_te = missing_pct(train_raw, cols), missing_pct(test_raw, cols)


def treatment(c):
    if c == TARGET:
        return "rows without a target are dropped"
    if c in CAT_FEATURES:
        return f"StringIndexer handleInvalid='{idx_m.getHandleInvalid()}'"
    return "median imputation (Imputer fitted on training rows only)"


missing_df = (pd.DataFrame([{"column": c, "train_missing_pct": round(miss_tr[c], 3),
                             "test_missing_pct": round(miss_te[c], 3), "treatment": treatment(c)}
                            for c in cols])
              .sort_values("train_missing_pct", ascending=False).reset_index(drop=True))
show(missing_df, "Table 2.0b  Missing values (% of rows) and how each column is handled")
missing_df.to_csv(RES_DIR / "task2_missing_values.csv", index=False)

train_raw = train_raw.dropna(subset=[TARGET])
test_raw  = test_raw.dropna(subset=[TARGET])
print(f"rows dropped for a missing target: train {n_train - train_raw.count():,}, "
      f"test {n_test - test_raw.count():,}")

Table 2.0b  Missing values (% of rows) and how each column is handled


,column,train_missing_pct,test_missing_pct,treatment
0,TOBS,21.771,25.966,median imputation (Imputer fitted on training ...
1,SNWD,11.740,23.252,median imputation (Imputer fitted on training ...
2,SNOW,9.253,22.112,median imputation (Imputer fitted on training ...
3,TMAX_lag2,1.707,2.215,median imputation (Imputer fitted on training ...
4,PRCP,1.085,1.564,median imputation (Imputer fitted on training ...
5,TMAX_lag1,0.886,1.160,median imputation (Imputer fitted on training ...
6,DTR,0.272,0.289,median imputation (Imputer fitted on training ...
7,TMIN,0.272,0.289,median imputation (Imputer fitted on training ...
8,TMIN_roll7,0.042,0.060,median imputation (Imputer fitted on training ...
9,state_code,0.000,0.000,StringIndexer handleInvalid='keep'


rows dropped for a missing target: train 0, test 0


### 2.0.4 The target variable and the persistence baseline
**Target:** `label_TMAX_next` is tomorrow's maximum temperature at the same station, in °C.

In [7]:
# ==============================================================================
#   Target profile and the persistence baseline
## AI: target profile and baseline written with AI assistance
# ==============================================================================
def target_profile(df):
    r = df.select(F.count(TARGET).alias("rows"), F.mean(TARGET).alias("mean"), F.stddev(TARGET).alias("sd"),
                  F.min(TARGET).alias("min"), F.percentile_approx(TARGET, [0.01, 0.5, 0.99], 10000).alias("p"),
                  F.max(TARGET).alias("max"), F.skewness(TARGET).alias("skew")).first()
    return {"rows": int(r["rows"]), "mean": r["mean"], "sd": r["sd"], "min": r["min"], "p01": r["p"][0],
            "median": r["p"][1], "p99": r["p"][2], "max": r["max"], "skew": r["skew"]}


TARGET_PROFILE = {"train": target_profile(train_raw), "test": target_profile(test_raw)}
show(pd.DataFrame(TARGET_PROFILE).T.round(3), f"Table 2.0c  Target {TARGET} (deg C)")

corr = sorted(task1["feature_target_correlation"].items(), key=lambda kv: -abs(kv[1]))
print("Strongest correlations with the target (Task 1, Table T1.2):",
      ", ".join(f"{k} {v:+.3f}" for k, v in corr[:5]))

err = F.col(TARGET) - F.col("TMAX")                        # persistence: prediction = today's TMAX
b = test_raw.select(F.sqrt(F.avg(err * err)).alias("rmse"), F.avg(F.abs(err)).alias("mae")).first()
print(f"Persistence baseline on the test years: RMSE {b['rmse']:.3f} deg C | MAE {b['mae']:.3f} deg C")

Table 2.0c  Target label_TMAX_next (deg C)


,rows,mean,sd,min,p01,median,p99,max,skew
train,2555512.0,17.274,11.950,-53.9,-12.2,18.9,37.8,50.0,-0.466
test,433788.0,17.932,11.931,-47.8,-11.7,19.4,38.3,47.8,-0.478


Strongest correlations with the target (Task 1, Table T1.2): TMAX +0.935, TMAX_roll7 +0.905, TOBS +0.886, TMAX_lag1 +0.883, TMAX_lag2 +0.858
Persistence baseline on the test years: RMSE 4.397 deg C | MAE 3.152 deg C


### 2.0.5 Feature preparation, fitted on training data only
Task 1's five stages are rebuilt **unfitted**, with exactly Task 1's settings:

1. `Imputer` (median)
2. `StringIndexer` (`state_code`)
3. `OneHotEncoder`
4. `VectorAssembler`
5. `StandardScaler` (unit variance; no centring, so the sparse one-hot vectors stay sparse)

These stages are placed **inside every model's Pipeline**. As a result:

In [8]:
# ==============================================================================
#  Feature preparation - Task 1's stages, rebuilt unfitted
# REF: Task 1 notebook section 10 (the same five stages and settings)
# ==============================================================================
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler


def make_preprocessing_stages():
    """New, UNFITTED copies of Task 1's five stages with Task 1's exact settings."""
    return [
        Imputer(inputCols=imp_m.getInputCols(), outputCols=imp_m.getOutputCols(),
                strategy=imp_m.getStrategy()),                                            # 1 median imputation
        StringIndexer(inputCols=idx_m.getInputCols(), outputCols=idx_m.getOutputCols(),
                      handleInvalid=idx_m.getHandleInvalid(),
                      stringOrderType=idx_m.getStringOrderType()),                          # 2 state -> index
        OneHotEncoder(inputCols=ohe_m.getInputCols(), outputCols=ohe_m.getOutputCols(),
                      dropLast=ohe_m.getDropLast(), handleInvalid=ohe_m.getHandleInvalid()),   # 3 index -> 0/1
        VectorAssembler(inputCols=va_t1.getInputCols(), outputCol=va_t1.getOutputCol(),
                        handleInvalid=va_t1.getHandleInvalid()),                            # 4 one vector
        StandardScaler(inputCol=scaler_m.getInputCol(), outputCol=scaler_m.getOutputCol(),
                       withMean=scaler_m.getWithMean(), withStd=scaler_m.getWithStd()),   # 5 unit variance
    ]


# compatibility check: fit the rebuilt stages on the training years and compare with Task 1's saved model
t0 = time.perf_counter()
prep_check = Pipeline(stages=make_preprocessing_stages()).fit(train_raw)
PREP_FIT_S = time.perf_counter() - t0
one_row = train_raw.limit(1)
N_FEATURES = prep_check.transform(one_row).first()[FEATURES_COL].size
n_feat_t1  = t1_prep.transform(one_row).first()[FEATURES_COL].size
sur_t1 = t1_prep.stages[0].surrogateDF.first().asDict()
sur_t2 = prep_check.stages[0].surrogateDF.first().asDict()
PREP_CHECK = {
    "stages": [type(s).__name__ for s in prep_check.stages],
    "feature_vector_size": N_FEATURES, "task1_feature_vector_size": n_feat_t1,
    "max_median_difference": max(abs(sur_t1[c] - sur_t2[c]) for c in sur_t1),
    "max_scaler_std_difference": float(np.max(np.abs(t1_prep.stages[4].std.toArray()
                                                     - prep_check.stages[4].std.toArray()))),
    "same_state_index": t1_prep.stages[1].labelsArray == prep_check.stages[1].labelsArray,
    "fit_time_s": round(PREP_FIT_S, 2),
}
for k, v in PREP_CHECK.items():
    print(f"{k:<28}: {v}")
assert N_FEATURES == n_feat_t1, "The rebuilt pipeline does not produce Task 1's feature vector"

stages                      : ['ImputerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel']
feature_vector_size         : 54
task1_feature_vector_size   : 54
max_median_difference       : 0.015051900466503903
max_scaler_std_difference   : 1.8669510382096632e-12
same_state_index            : True
fit_time_s                  : 34.4


### 2.0.6 Training, tuning and test sets
**The train/test split stays exactly as Task 1 made it**, by year:

- **training:** 1970–2015;
- **test:** 2016–2023.
 

In [9]:
# ==============================================================================
# Training, tuning and test sets
## AI: hash-based sampling and the year-block fold column design with AI assistance
# ==============================================================================
MODEL_COLS = KEY_COLS + CAT_FEATURES + NUM_FEATURES + [TARGET]
row_hash   = F.abs(F.xxhash64("ID", "date"))                # EXPLAIN: same number for a row on every computer

train_df = train_raw.select(*MODEL_COLS)
if FINAL_TRAIN_PCT < 100:
    train_df = train_df.filter(row_hash % 100 < FINAL_TRAIN_PCT)
tune_base = train_raw.select(*MODEL_COLS).filter(row_hash % 100 < TUNE_PCT)

# three blocks of consecutive years with about the same number of rows each (NUM_FOLDS must be 3)
assert NUM_FOLDS == 3, "the fold rule below makes exactly three year blocks"
cut1, cut2 = tune_base.select(F.percentile_approx("YEAR", [1 / 3, 2 / 3], 10000)).first()[0]
fold_expr  = F.when(F.col("YEAR") <= cut1, 0).when(F.col("YEAR") <= cut2, 1).otherwise(2)

tune_df  = tune_base.withColumn("fold", fold_expr.cast("int")).cache()
train_df = train_df.cache()
test_df  = test_raw.select(*MODEL_COLS).cache()
N_TUNE, N_TRAIN, N_TEST = tune_df.count(), train_df.count(), test_df.count()   # EXPLAIN: fills the caches

fold_tbl = (tune_df.groupBy("fold")
                   .agg(F.min("YEAR").alias("first_year"), F.max("YEAR").alias("last_year"),
                        F.count(F.lit(1)).alias("rows"))
                   .orderBy("fold").toPandas())
show(fold_tbl, "Table 2.0d  Cross-validation folds (blocks of consecutive years)")
print(f"training set : {N_TRAIN:,} rows, years {yr_tr[0]}-{yr_tr[1]} ({FINAL_TRAIN_PCT}% of the Task 1 training set)")
print(f"tuning sample: {N_TUNE:,} rows ({TUNE_PCT}% of the training rows, chosen by a hash of ID + date)")
print(f"test set     : {N_TEST:,} rows, years {yr_te[0]}-{yr_te[1]} - used once, in section 4")

Table 2.0d  Cross-validation folds (blocks of consecutive years)


,fold,first_year,last_year,rows
0,0,1970,1985,87683
1,1,1986,2001,88056
2,2,2002,2015,80116


training set : 2,555,512 rows, years 1970-2015 (100% of the Task 1 training set)
tuning sample: 255,855 rows (10% of the training rows, chosen by a hash of ID + date)
test set     : 433,788 rows, years 2016-2023 - used once, in section 4


## 2.1 Four regression algorithms
| # | Model (Spark MLlib) | Family | Why it suits this dataset |
|---|---|---|---|
| 1 | `LinearRegression` | Linear | Task 1 measured r = 0.93 between today's and tomorrow's `TMAX`, so most of the signal is linear. This is the cheapest and most interpretable model, and it sets the bar the others must clear. Regularisation tests whether the strongly correlated inputs need shrinking. `TMAX`, `TMAX_lag1` and `TMAX_roll7` overlap, and `DTR` is exactly `TMAX − TMIN`. |
| 2 | `RandomForestRegressor` | Tree-based ensemble (bagging) | Weather effects depend on season, snow cover and state. For example, snow depth matters only in winter. Trees learn these thresholds and interactions without being told. Bagged trees are trained in parallel across the cores, and they need no scaling and assume no distribution, which suits the skewed `PRCP` and `SNOW`. |
| 3 | `FMRegressor` | Distance / kernel / neural (factorisation machine) | A factorisation machine is a factorised degree-2 polynomial kernel. It models every pairwise interaction, such as state × season or snow × season, through low-rank factors. Its cost is linear in the 54 mostly sparse inputs, so it scales to 2.5 million rows where a kernel SVM would not. It needs scaled inputs, which Task 1's `StandardScaler` provides. |
| 4 | `GBTRegressor` | Free choice: tree-based boosting | Boosting fits each new tree to the errors of the trees before it, so it focuses on the days persistence gets wrong, such as fronts and cold-air outbreaks. It is often the most accurate model on tabular data. Its trees are built one after another, so it is the test case for accuracy against compute cost. |

A single `DecisionTreeRegressor` is **not** used as a separate model. It is the building block of both the forest and the boosted trees, so it would add no new family 

In [11]:
# ==============================================================================
#  The four models and their parameter grids
# REF: Day06_Regression_LiveCoding_Solutions_Colab > cell-9 [Day 6 Session A]: LinearRegression,
#      GBTRegressor and ParamGridBuilder(gbt.maxDepth, [3, 5]).
#      Changed: four families instead of two; each grid searches the two parameters that control that
#      model's bias-variance trade-off; Task 1's preprocessing sits inside every Pipeline.
# ==============================================================================
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, FMRegressor, GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

lr  = LinearRegression(featuresCol=FEATURES_COL, labelCol=TARGET, maxIter=100)
rf  = RandomForestRegressor(featuresCol=FEATURES_COL, labelCol=TARGET, seed=SEED,
                            featureSubsetStrategy="onethird",
                            maxMemoryInMB=64, cacheNodeIds=True)    # EXPLAIN: 512 ran out of Java heap (12 tasks x 2 fits)
fm  = FMRegressor(featuresCol=FEATURES_COL, labelCol=TARGET, seed=SEED,
                  stepSize=0.01, regParam=0.0, solver="adamW")      # EXPLAIN: step sizes >= 0.1 diverged in a pilot run
gbt = GBTRegressor(featuresCol=FEATURES_COL, labelCol=TARGET, seed=SEED,
                   stepSize=0.1, subsamplingRate=0.8,
                   maxMemoryInMB=64, cacheNodeIds=True, checkpointInterval=10)

MODELS = {
    "LinearRegression": {
        "family": "Linear",
        "estimator": lr,
        # ordinary least squares once, then ridge (0), elastic net (0.5) and lasso (1) at two strengths
        "grid": [{lr.regParam: 0.0, lr.elasticNetParam: 0.0}]
                + ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1])
                                    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]).build(),
        "reason": "r = 0.93 between today's and tomorrow's TMAX, so most of the signal is linear; "
                  "cheapest, interpretable baseline the other models must beat",
    },
    "RandomForestRegressor": {
        "family": "Tree-based ensemble (bagging)",
        "estimator": rf,
        "grid": ParamGridBuilder().addGrid(rf.numTrees, [20, 50]).addGrid(rf.maxDepth, [8, 12]).build(),
        "reason": "learns season x snow x state thresholds without a set form; trees train in parallel "
                  "and need no scaling for skewed PRCP/SNOW",
    },
    "FMRegressor": {
        "family": "Distance / kernel / neural (factorisation machine)",
        "estimator": fm,
        # maxIter halved from [100, 200]: 200 iterations did not finish within an hour on the 12-core laptop
        "grid": ParamGridBuilder().addGrid(fm.factorSize, [4, 8]).addGrid(fm.maxIter, [50, 100]).build(),
        "reason": "factorised degree-2 polynomial kernel: models all pairwise interactions "
                  "(state x season) at a cost linear in the 54 sparse inputs",
    },
    "GBTRegressor": {
        "family": "Free choice: tree-based boosting",
        "estimator": gbt,
        "grid": ParamGridBuilder().addGrid(gbt.maxDepth, [3, 5]).addGrid(gbt.maxIter, [30, 60]).build(),
        "reason": "each tree corrects the previous trees' errors, targeting the days persistence gets "
                  "wrong; sequential, so it tests accuracy against compute cost",
    },
}
if QUICK_TEST:                                             # smoke test only: two combinations per model
    for spec in MODELS.values():
        spec["grid"] = spec["grid"][:2]


def searched_values(grid):
    """{parameter: sorted list of values searched} for one grid."""
    out = {}
    for pm in grid:
        for p, v in pm.items():
            out.setdefault(p.name, set()).add(v)
    return {k: sorted(v) for k, v in out.items()}


selection_df = pd.DataFrame([{
    "model": n, "family": s["family"], "reason": s["reason"],
    "grid": "; ".join(f"{k} {v}" for k, v in searched_values(s["grid"]).items()),
    "combinations": len(s["grid"]), "cv_folds": NUM_FOLDS, "fits": len(s["grid"]) * NUM_FOLDS,
} for n, s in MODELS.items()])
show(selection_df[["model", "family", "grid", "combinations", "fits"]], "Table 2.1  Models and grids searched")
selection_df.to_csv(RES_DIR / "task2_model_selection.csv", index=False)
print(f"{len(MODELS)} models from {selection_df['family'].nunique()} families; "
      f"{selection_df['fits'].sum()} cross-validation fits in total, then {len(MODELS)} final refits")

Table 2.1  Models and grids searched


,model,family,grid,combinations,fits
0,LinearRegression,Linear,"regParam [0.0, 0.01, 0.1]; elasticNetParam [0....",7,21
1,RandomForestRegressor,Tree-based ensemble (bagging),"numTrees [20, 50]; maxDepth [8, 12]",4,12
2,FMRegressor,Distance / kernel / neural (factorisation mach...,"factorSize [4, 8]; maxIter [50, 100]",4,12
3,GBTRegressor,Free choice: tree-based boosting,"maxDepth [3, 5]; maxIter [30, 60]",4,12


4 models from 4 families; 57 cross-validation fits in total, then 4 final refits


## 2.1.1 Grid search and cross-validation

**Two different times are recorded:**

- **Total tuning time:** wall-clock time of `CrossValidator.fit`. This covers every combination × fold, plus the refit that `CrossValidator` makes on the tuning sample.
- **Final fit time:** wall-clock time to refit the winning pipeline, preprocessing included, on the full training set. This is the cost of producing the model that is evaluated in section 4.

Each model has its own cell. Results are checkpointed, so if the laptop crashes, a re-run loads the models that already finished.

In [12]:
# ==============================================================================
#  Tune -> refit -> save, for one model at a time
# REF: Day05_LiveCoding_Colab > Block 3 (cell-9) [Day 5]: CrossValidator(estimator, estimatorParamMaps,
#      evaluator, numFolds=3, parallelism=2, seed=42).fit(train); Slide36_LiveCoding_Colab > Step 6
#      (best-parameter extraction).
#      Changed: Pipeline (preprocessing + model) as the estimator, year-block foldCol, the whole grid
#      recorded (avgMetrics + stdMetrics), refit on the full training set, checkpoint after each model.
#
# ==============================================================================
cv_evaluator = RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="rmse")
TUNED, FITTED = {}, {}


def run_model(name):
    spec = MODELS[name]
    ckpt, model_path = CKPT_DIR / f"{name}.json", MODEL_DIR / name

    if REUSE_SAVED and ckpt.exists() and (model_path / "metadata").exists():      # restart after a crash
        TUNED[name] = json.loads(ckpt.read_text())
        FITTED[name] = PipelineModel.load(spark_path(model_path))
        print(f"[REUSED] {name}: finished {TUNED[name]['saved_at']} on {TUNED[name]['executor']}")
        return

    grid = spec["grid"]
    cv = CrossValidator(estimator=Pipeline(stages=make_preprocessing_stages() + [spec["estimator"]]),
                        estimatorParamMaps=grid, evaluator=cv_evaluator, numFolds=NUM_FOLDS,
                        foldCol="fold", parallelism=CV_PARALLELISM, seed=SEED)
    print(f"[{time.strftime('%H:%M:%S')}] {name}: {len(grid)} combinations x {NUM_FOLDS} folds "
          f"on {N_TUNE:,} rows ...", flush=True)
    t0 = time.perf_counter()
    cv_model = cv.fit(tune_df)
    tuning_s = time.perf_counter() - t0

    avg = [float(x) for x in cv_model.avgMetrics]
    std = [float(x) for x in (getattr(cv_model, "stdMetrics", None) or [float("nan")] * len(avg))]
    best_i = int(np.nanargmin(avg))
    order = np.argsort(avg)
    ranks = {int(i): r + 1 for r, i in enumerate(order)}
    cv_grid = [{**{p.name: v for p, v in pm.items()}, "mean_cv_rmse": a, "std_cv_rmse": s, "rank": ranks[i]}
               for i, (pm, a, s) in enumerate(zip(grid, avg, std))]
    best_params = {p.name: v for p, v in grid[best_i].items()}
    print(f"[{time.strftime('%H:%M:%S')}] {name}: tuning done in {tuning_s:.0f} s; best CV RMSE "
          f"{avg[best_i]:.4f}; refitting on {N_TRAIN:,} rows ...", flush=True)

    final = Pipeline(stages=make_preprocessing_stages() + [spec["estimator"].copy(grid[best_i])])
    t0 = time.perf_counter()
    model = final.fit(train_df)
    fit_s = time.perf_counter() - t0
    model.write().overwrite().save(spark_path(model_path))

    TUNED[name] = {
        "model": name, "family": spec["family"], "cv_metric": "rmse", "cv_folds": NUM_FOLDS,
        "fold_rule": "blocks of consecutive years (foldCol)", "combinations": len(grid),
        "searched": searched_values(grid), "tuning_rows": N_TUNE, "cv_grid": cv_grid,
        "best_params": best_params, "best_cv_rmse": avg[best_i], "best_cv_rmse_std": std[best_i],
        "tuning_time_s": round(tuning_s, 2), "final_fit_rows": N_TRAIN, "final_fit_time_s": round(fit_s, 2),
        "model_path": rel(model_path), "executor": EXECUTOR_STR, "quick_test": QUICK_TEST,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    ckpt.write_text(json.dumps(TUNED[name], indent=2, default=str))
    FITTED[name] = model
    print(f"          best: {', '.join(f'{k}={v}' for k, v in best_params.items())} | tuning {tuning_s:.1f} s | "
          f"final fit {fit_s:.1f} s -> saved to {rel(model_path)}")


print("run_model() ready - run the four model cells below one at a time")

run_model() ready - run the four model cells below one at a time


### 2.1.2 Model 1 — Linear Regression

In [13]:
run_model("LinearRegression")

[REUSED] LinearRegression: finished 2026-10-18 19:09:16 on local[*], 12 cores, driver 4g, 32 shuffle partitions, CV parallelism 2


### 2.1.3 Model 2 — Random Forest Regressor

In [14]:
run_model("RandomForestRegressor")

[REUSED] RandomForestRegressor: finished 2026-10-19 15:37:45 on local[*], 12 cores, driver 4g, 32 shuffle partitions, CV parallelism 2


### 2.1.4 Model 3 — Factorisation Machine Regressor

In [15]:
run_model("FMRegressor")

[17:04:08] FMRegressor: 4 combinations x 3 folds on 255,855 rows ...
[17:16:07] FMRegressor: tuning done in 720 s; best CV RMSE 4.0127; refitting on 2,555,512 rows ...
          best: factorSize=8, maxIter=100 | tuning 719.7 s | final fit 1082.6 s -> saved to models/task2/FMRegressor


### 2.1.5 Model 4 — Gradient-Boosted Tree Regressor

In [16]:
run_model("GBTRegressor")

[17:36:19] GBTRegressor: 4 combinations x 3 folds on 255,855 rows ...
[17:54:40] GBTRegressor: tuning done in 1100 s; best CV RMSE 3.8237; refitting on 2,555,512 rows ...
          best: maxDepth=5, maxIter=60 | tuning 1100.1 s | final fit 661.8 s -> saved to models/task2/GBTRegressor
